In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG   = "clutchlytics"
SCHEMA    = "bronze"
VOLUME    = "nhl_raw"
TABLE     = f"{CATALOG}.{SCHEMA}.raw_nhl_scoreboard"
 
# ← Update FILE_NAME for each new round upload
FILE_NAME = "round_1_scoreboard_raw.json"
 
FILE_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{FILE_NAME}"
 
print(f"Source : {FILE_PATH}")
print(f"Target : {TABLE}")

In [0]:
# ── CONFIRM FILE EXISTS ───────────────────────────────────────────────────────
 
files      = dbutils.fs.ls(f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}")
file_names = [f.name for f in files]
 
if FILE_NAME not in file_names:
    raise FileNotFoundError(
        f"'{FILE_NAME}' not found in Volume.\n"
        f"Files available: {file_names}"
    )
 
file_size = [f.size for f in files if f.name == FILE_NAME][0]
print(f"File confirmed: {FILE_NAME} ({file_size:,} bytes)")

In [0]:
# ── READ + PARSE JSON ────────────────────────────────────────────────────────
 
import json
from datetime import datetime, timezone
 
raw_text  = spark.read.text(FILE_PATH)
json_str  = "\n".join([row.value for row in raw_text.collect()])
payload   = json.loads(json_str)
 
# ── Extract meta envelope ──
meta        = payload.get("meta", {})
pulled_at   = meta.get("pulled_at")
season      = meta.get("season")
season_type = meta.get("season_type")
round_num   = meta.get("round")
source_url  = meta.get("source_url")
 
# ── Extract events ──
data   = payload.get("data", {})
events = data.get("events", [])
 
print(f"Meta:")
print(f"  season      : {season}")
print(f"  season_type : {season_type}")
print(f"  round       : {round_num}")
print(f"  pulled_at   : {pulled_at}")
print(f"\nEvents found: {len(events)}")
 
# Print raw featuredAthletes from first event so we can confirm parsing
first_event = events[0] if events else {}
first_comp  = first_event.get("competitions", [{}])[0]
first_status = first_comp.get("status", {})
print(f"\nfeaturedAthletes (first event, raw):")
print(json.dumps(first_status.get("featuredAthletes", []), indent=2)[:1500])
 
# COMMAND ----------
 
# ── HELPER FUNCTIONS ──────────────────────────────────────────────────────────
 
def get_featured(featured_list, name_key):
    """
    Match on the camelCase 'name' field e.g. 'winningGoalie', 'firstStar'.
    featuredAthletes lives under status{} not competitions{} directly.
    """
    return next(
        (a for a in featured_list if a.get("name") == name_key), {}
    )
 
def featured_id(entry):
    """
    ESPN athlete ID. Available at Bronze — no Silver name resolution needed.
    Prefer athlete.id, fall back to playerId.
    """
    athlete_id = entry.get("athlete", {}).get("id")
    if athlete_id:
        return str(athlete_id)
    player_id = entry.get("playerId")
    return str(player_id) if player_id else None
 
def featured_name(entry):
    athlete = entry.get("athlete", {})
    return athlete.get("displayName") or athlete.get("fullName")
 
def featured_stat(entry, stat_name):
    """
    Extract a stat by name from the statistics array.
    Returns None for skater entries that have goalie fields set to 0/.000
    so we don't pollute goalie stats with skater placeholders.
    """
    stats = entry.get("statistics", [])
    match = next((s for s in stats if s.get("name") == stat_name), {})
    val   = match.get("displayValue")
    # Skaters get goalie fields set to 0 or .000 — treat as null
    if val in (None, "", "0", ".000"):
        return None
    return val
 
def get_team(competitor):
    return competitor.get("team", {})
 
def get_linescore(competitor, period_num):
    ls    = competitor.get("linescores", [])
    match = next((l for l in ls if l.get("period") == period_num), {})
    return match.get("value")
 
def get_record(competitor, rec_type):
    records = competitor.get("records", [])
    match   = next((r for r in records if r.get("type") == rec_type), {})
    return match.get("summary")

In [0]:
# ── FLATTEN TO ONE ROW PER COMPETITION ───────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
rows        = []
 
for event in events:
    event_id   = event.get("id")
    event_name = event.get("name")
    short_name = event.get("shortName")
    game_date  = event.get("date")
 
    for comp in event.get("competitions", []):
        comp_id = comp.get("id")
 
        # ── Status ──
        # featuredAthletes lives HERE — under status{}, not competitions{}
        status      = comp.get("status", {})
        status_type = status.get("type", {})
        featured    = status.get("featuredAthletes", [])
 
        status_state       = status_type.get("state")
        status_description = status_type.get("description")
        status_detail      = status_type.get("detail")
        status_name        = status_type.get("name")
        period             = status.get("period")
        completed          = status_type.get("completed", False)
 
        # ── Competitors — home and away ──
        competitors  = comp.get("competitors", [])
        home         = next((c for c in competitors if c.get("homeAway") == "home"), {})
        away         = next((c for c in competitors if c.get("homeAway") == "away"), {})
        home_team_id = home.get("id")
        away_team_id = away.get("id")
 
        # ── Series context ──
        series             = comp.get("series", {})
        series_competitors = series.get("competitors", [])
 
        home_series_wins = next(
            (s.get("wins") for s in series_competitors if s.get("id") == home_team_id), None
        )
        away_series_wins = next(
            (s.get("wins") for s in series_competitors if s.get("id") == away_team_id), None
        )
 
        # ── Featured athletes ──
        # Match on camelCase 'name' field
        win_g  = get_featured(featured, "winningGoalie")
        lose_g = get_featured(featured, "losingGoalie")
        star1  = get_featured(featured, "firstStar")
        star2  = get_featured(featured, "secondStar")
        star3  = get_featured(featured, "thirdStar")
 
        rows.append({
            # ── Event ──
            "event_id":               event_id,
            "comp_id":                comp_id,
            "event_name":             event_name,
            "short_name":             short_name,
            "game_date":              game_date,
 
            # ── Status ──
            "status_state":           status_state,
            "status_description":     status_description,
            "status_detail":          status_detail,
            "status_name":            status_name,
            "period":                 period,
            "completed":              completed,
 
            # ── Home team ──
            "home_team_id":           home_team_id,
            "home_team_abbr":         get_team(home).get("abbreviation"),
            "home_team_name":         get_team(home).get("displayName"),
            "home_score":             home.get("score"),
            "home_winner":            home.get("winner"),
            "home_p1":                get_linescore(home, 1),
            "home_p2":                get_linescore(home, 2),
            "home_p3":                get_linescore(home, 3),
            "home_ot":                get_linescore(home, 4),
            "home_record_total":      get_record(home, "total"),
            "home_record_home":       get_record(home, "home"),
            "home_record_road":       get_record(home, "road"),
 
            # ── Away team ──
            "away_team_id":           away_team_id,
            "away_team_abbr":         get_team(away).get("abbreviation"),
            "away_team_name":         get_team(away).get("displayName"),
            "away_score":             away.get("score"),
            "away_winner":            away.get("winner"),
            "away_p1":                get_linescore(away, 1),
            "away_p2":                get_linescore(away, 2),
            "away_p3":                get_linescore(away, 3),
            "away_ot":                get_linescore(away, 4),
            "away_record_total":      get_record(away, "total"),
            "away_record_home":       get_record(away, "home"),
            "away_record_road":       get_record(away, "road"),
 
            # ── Series ──
            "series_summary":         series.get("summary"),
            "series_completed":       series.get("completed"),
            "series_total_games":     series.get("totalCompetitions"),
            "series_type":            series.get("type"),
            "home_series_wins":       home_series_wins,
            "away_series_wins":       away_series_wins,
 
            # ── Winning goalie ──
            "winning_goalie_id":      featured_id(win_g),
            "winning_goalie_name":    featured_name(win_g),
            "winning_goalie_saves":   featured_stat(win_g, "saves"),
            "winning_goalie_sv_pct":  featured_stat(win_g, "savePct"),
 
            # ── Losing goalie ──
            "losing_goalie_id":       featured_id(lose_g),
            "losing_goalie_name":     featured_name(lose_g),
            "losing_goalie_saves":    featured_stat(lose_g, "saves"),
            "losing_goalie_sv_pct":   featured_stat(lose_g, "savePct"),
 
            # ── Three stars ──
            "first_star_id":          featured_id(star1),
            "first_star_name":        featured_name(star1),
            "second_star_id":         featured_id(star2),
            "second_star_name":       featured_name(star2),
            "third_star_id":          featured_id(star3),
            "third_star_name":        featured_name(star3),
 
            # ── Season / playoff context ──
            "season":                 season,
            "season_type":            season_type,
            "round":                  round_num,
 
            # ── Ingestion metadata ──
            "source_file":            FILE_NAME,
            "source_url":             source_url,
            "pulled_at":              pulled_at,
            "ingested_at":            ingested_at,
        })
 
print(f"Rows built: {len(rows)}")
print(f"\nSample — featured athletes first 5 games:")
print(f"  {'short_name':<20} {'win_goalie':<25} {'saves':<6} {'sv_pct':<8} {'1st_star'}")
for r in rows[:5]:
    print(f"  {str(r['short_name']):<20} {str(r['winning_goalie_name']):<25} "
          f"{str(r['winning_goalie_saves']):<6} {str(r['winning_goalie_sv_pct']):<8} "
          f"{str(r['first_star_name'])}")

In [0]:
# ── WRITE TO DELTA ────────────────────────────────────────────────────────────
# MERGE on event_id + comp_id — safe for re-uploads and future rounds.
 
scoreboard_df = spark.createDataFrame(rows)
 
table_exists = spark.catalog.tableExists(TABLE)
 
if not table_exists:
    (
        scoreboard_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(TABLE)
    )
    print(f"Table created: {TABLE}")
 
else:
    scoreboard_df.createOrReplaceTempView("new_scoreboard")
 
    spark.sql(f"""
        MERGE INTO {TABLE} AS target
        USING new_scoreboard AS source
        ON  target.event_id = source.event_id
        AND target.comp_id  = source.comp_id
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    print(f"Merged into existing table: {TABLE}")

In [0]:
# ── VALIDATE ─────────────────────────────────────────────────────────────────
 
result = spark.sql(f"""
    SELECT
        event_id,
        short_name,
        game_date,
        home_team_abbr,
        home_score,
        away_team_abbr,
        away_score,
        home_p1, home_p2, home_p3, home_ot,
        series_summary,
        home_series_wins,
        away_series_wins,
        winning_goalie_name,
        winning_goalie_saves,
        winning_goalie_sv_pct,
        first_star_name,
        completed,
        round
    FROM {TABLE}
    ORDER BY game_date
""")
 
total = result.count()
print(f"Total rows in {TABLE}: {total}")
result.show(50, truncate=False)

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_rows,
        COUNT(DISTINCT round)                                       AS rounds,
        COUNT(CASE WHEN completed = true  THEN 1 END)              AS completed_games,
        COUNT(CASE WHEN completed = false THEN 1 END)              AS incomplete_games,
        COUNT(CASE WHEN event_id IS NULL  THEN 1 END)              AS null_event_ids,
        COUNT(CASE WHEN home_team_id IS NULL THEN 1 END)           AS null_home_teams,
        COUNT(CASE WHEN away_team_id IS NULL THEN 1 END)           AS null_away_teams,
        COUNT(CASE WHEN winning_goalie_id IS NOT NULL THEN 1 END)  AS games_with_goalie_id,
        COUNT(CASE WHEN winning_goalie_saves IS NOT NULL THEN 1 END) AS games_with_goalie_saves,
        COUNT(CASE WHEN first_star_id IS NOT NULL THEN 1 END)      AS games_with_stars,
        COUNT(CASE WHEN home_ot IS NOT NULL THEN 1 END)            AS ot_games,
        MIN(game_date)                                             AS earliest_game,
        MAX(game_date)                                             AS latest_game
    FROM {TABLE}
""")
 
print("Sanity checks:")
checks.show(truncate=False)